# 4 — Gráficos: Perfil dos Estudantes

Todos os gráficos usam o **último ano disponível** nos dados.

**Seção 1 — Perfil geral:**

| Código | Tipo | Descrição |
|--------|------|-----------|
| 10 | Pirâmide etária | Faixa etária × sexo |
| 11 | Barras horizontais | Distribuição por cor/raça |
| 12 | Barras horizontais | Distribuição por renda familiar |
| 13 | Pizza (donut) | Distribuição por turno |

**Seção 2 — Heatmaps de evasão:**

| Código | Tipo | Descrição |
|--------|------|-----------|
| 14 | Heatmap | Evasão — Faixa Etária × Renda Familiar |
| 15 | Heatmap | Evasão — Sexo × Turno |
| 16 | Heatmap | Evasão — Cor/Raça × Renda Familiar |

**Seção 3 — Heatmaps de retenção:**

| Código | Tipo | Descrição |
|--------|------|-----------|
| 17 | Heatmap | Retenção — Faixa Etária × Renda Familiar |
| 18 | Heatmap | Retenção — Sexo × Turno |
| 19 | Heatmap | Retenção — Cor/Raça × Renda Familiar |

**Seção 4 — Visão consolidada:**

| Código | Tipo | Descrição |
|--------|------|-----------|
| 20 | Heatmap | Situação (Evasão/Retenção/Conclusão) × Perfil Sociodemográfico |


### 4.1. Importações e configurações

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

ORDEM_ETARIA = [
    "Menor de 14 anos", "15 a 19 anos", "20 a 24 anos",
    "25 a 29 anos", "30 a 34 anos", "35 a 39 anos",
    "40 a 44 anos", "45 a 49 anos", "50 a 54 anos",
    "55 a 59 anos", "Maior de 60 anos",
]
ORDEM_RENDA = [
    "0<RFP<=0,5", "0,5<RFP<=1", "1<RFP<=1,5",
    "1,5<RFP<=2,5", "2,5<RFP<=3,5", "RFP>3,5", "Não declarada",
]
CORES_SEXO = {"M": "#2196F3", "F": "#E91E63"}

print("Imports OK")


Imports OK


### 4.2. Carga dos dados

In [2]:
df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

# Usa apenas o último ano disponível
ultimo_ano = int(df_completo["Ano"].max())
df = df_completo[df_completo["Ano"] == ultimo_ano].copy()

# Mantém apenas as categorias presentes nos dados
faixas_presentes = [f for f in ORDEM_ETARIA if f in df["Faixa Etária"].unique()]
rendas_presentes = [r for r in ORDEM_RENDA  if r in df["Renda Familiar"].unique()]

print(f"Último ano: {ultimo_ano} | Shape: {df.shape}")
df.head(3)


Último ano: 2024 | Shape: (1812, 19)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Concluinte_Previsto
8633,2024,Em curso,Branca,116719445,3069953,2027-12-30,2024-02-19,2024-02-01,Controle e Processos Industriais,15 a 19 anos,2024-02-25,Técnico em Eletrônica,Não declarada,M,Ingressante,Elétrica,Técnico-Integrado,Matutino,False
8634,2024,Em curso,Branca,116719459,3069953,2027-12-30,2024-02-19,2024-02-01,Controle e Processos Industriais,15 a 19 anos,2024-02-25,Técnico em Eletrônica,"0,5<RFP<=1",M,Ingressante,Elétrica,Técnico-Integrado,Matutino,False
8635,2024,Em curso,Parda,116719439,3069953,2027-12-30,2024-02-19,2024-02-01,Controle e Processos Industriais,15 a 19 anos,2024-02-25,Técnico em Eletrônica,"1<RFP<=1,5",M,Ingressante,Elétrica,Técnico-Integrado,Matutino,False


### 4.4. Seção 1 — Perfil geral (gráficos 10–13)


### 4.4. Seção 1 — Perfil no último ano

In [3]:
# ── Gráfico 10: Pirâmide etária por sexo (último ano) ────────────────────────
# A pirâmide usa go.Bar com valores negativos no lado esquerdo (primeiro sexo)
# e positivos no lado direito (segundo sexo). O eixo X exibe apenas valores
# absolutos para não confundir o leitor com os sinais negativos.

piramide_df = (
    df.groupby(["Faixa Etária", "Sexo"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

# Garante a ordem cronológica das faixas no eixo Y
piramide_df["Faixa Etária"] = pd.Categorical(
    piramide_df["Faixa Etária"],
    categories=faixas_presentes,
    ordered=True,
)
piramide_df = piramide_df.sort_values("Faixa Etária")

sexos = piramide_df["Sexo"].dropna().unique().tolist()

fig_10 = go.Figure()

for i, sexo in enumerate(sexos):
    subset = piramide_df[piramide_df["Sexo"] == sexo]
    multiplicador = -1 if i == 0 else 1   # primeiro sexo → lado esquerdo
    valores = subset["Qtd"] * multiplicador

    fig_10.add_trace(go.Bar(
        name=sexo,
        y=subset["Faixa Etária"].astype(str),
        x=valores,
        orientation="h",
        marker_color=CORES_SEXO.get(sexo, "#9E9E9E"),
        text=subset["Qtd"].astype(str),   # valor absoluto no rótulo
        textposition="inside",
    ))

valor_max = int(piramide_df["Qtd"].max())
passo     = max(1, valor_max // 5)
ticks     = list(range(-valor_max, valor_max + passo, passo))

fig_10.update_layout(
    title=f"10 — Pirâmide Etária por Sexo ({ultimo_ano})",
    barmode="relative",
    xaxis=dict(
        tickvals=ticks,
        ticktext=[str(abs(v)) for v in ticks],
        title="Matrículas",
    ),
    yaxis=dict(title=""),
    legend=dict(orientation="h", y=-0.18),
)
fig_10.show()

In [4]:
# ── Gráfico 11: Distribuição por cor/raça — último ano (barras horizontais) ───

raca_df = df["Cor / Raça"].value_counts().reset_index()
raca_df.columns = ["Cor/Raça", "Qtd"]

fig_11 = px.bar(
    raca_df.sort_values("Qtd", ascending=True),
    x="Qtd",
    y="Cor/Raça",
    orientation="h",
    color="Qtd",
    color_continuous_scale="Purples",
    text_auto=True,
    title=f"11 — Distribuição por Cor/Raça ({ultimo_ano})",
    labels={"Qtd": "Matrículas"},
)
fig_11.update_layout(coloraxis_showscale=False)
fig_11.show()

In [5]:
# ── Gráfico 12: Distribuição por renda familiar — último ano (barras horizontais)
# Barras horizontais facilitam a leitura dos rótulos longos das faixas de renda.
# reindex() garante que as faixas apareçam na ordem lógica (menor → maior renda).

rendas_ultimo = [r for r in rendas_presentes if r in df["Renda Familiar"].unique()]

renda_df = (
    df["Renda Familiar"]
    .value_counts()
    .reindex(rendas_ultimo)
    .reset_index()
)
renda_df.columns = ["Renda", "Qtd"]

fig_12 = px.bar(
    renda_df.sort_values("Qtd", ascending=True),
    x="Qtd",
    y="Renda",
    orientation="h",
    color="Qtd",
    color_continuous_scale="Greens",
    text_auto=True,
    title=f"12 — Distribuição por Renda Familiar ({ultimo_ano})",
    labels={"Qtd": "Matrículas", "Renda": "Renda (SM per capita)"},
)
fig_12.update_layout(coloraxis_showscale=False)
fig_12.show()

In [6]:
# ── Gráfico 13: Distribuição por turno — último ano (pizza/donut) ─────────────

turno_df = df["Turno"].value_counts().reset_index()
turno_df.columns = ["Turno", "Qtd"]

fig_13 = px.pie(
    turno_df,
    names="Turno",
    values="Qtd",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f"13 — Distribuição por Turno ({ultimo_ano})",
)
fig_13.update_traces(textposition="outside", textinfo="percent+label")
fig_13.show()

### 4.5. Função auxiliar: heatmap_cruzado


In [7]:
def heatmap_cruzado(df, col_y, col_x, situacao, ordem_y=None, ordem_x=None):
    """
    Calcula a taxa de evasão ou retenção para cada combinação
    col_y × col_x e retorna um DataFrame pivotado para px.imshow.

    Parâmetros
    ----------
    df        : DataFrame do último ano
    col_y     : variável para o eixo Y (ex: "Faixa Etária")
    col_x     : variável para o eixo X (ex: "Renda Familiar")
    situacao  : "Evadidos" ou "Retido"
    ordem_y   : lista para ordenar as linhas
    ordem_x   : lista para ordenar as colunas
    """
    if situacao == "Evadidos":
        mask = df["Categoria da Situação"] == "Evadidos"
    else:
        mask = df["Situação de Matrícula"] == "Retido"

    contagem = (
        df[mask].groupby([col_y, col_x])["Código da Matricula"]
        .count().reset_index(name="N")
    )
    total = (
        df.groupby([col_y, col_x])["Código da Matricula"]
        .count().reset_index(name="Total")
    )
    merged = total.merge(contagem, on=[col_y, col_x], how="left").fillna(0)
    merged["Taxa (%)"] = (merged["N"] / merged["Total"] * 100).round(1)

    pivot = merged.pivot_table(
        index=col_y, columns=col_x, values="Taxa (%)", fill_value=np.nan
    )
    if ordem_y:
        pivot = pivot.reindex([c for c in ordem_y if c in pivot.index])
    if ordem_x:
        pivot = pivot[[c for c in ordem_x if c in pivot.columns]]

    return pivot


print("heatmap_cruzado() definida")


heatmap_cruzado() definida


### 4.6. Seção 2 — Heatmaps de evasão (gráficos 14–16)


In [8]:
# ── Gráfico 14: Evasão — Faixa Etária × Renda Familiar ───────────────────────
# Cada célula: % de evadidos entre os estudantes daquela faixa etária E renda.
# NaN (branco) = nenhuma matrícula nessa combinação.

pivot_14 = heatmap_cruzado(
    df, "Faixa Etária", "Renda Familiar", "Evadidos",
    ordem_y=ORDEM_ETARIA, ordem_x=ORDEM_RENDA,
)
fig_14 = px.imshow(
    pivot_14, text_auto=".1f", color_continuous_scale="Reds",
    labels={"color": "Evasão (%)"},
    title=f"14 — Taxa de Evasão: Faixa Etária × Renda Familiar ({ultimo_ano})",
    aspect="auto",
)
fig_14.update_layout(
    xaxis_title="Renda Familiar (SM per capita)",
    yaxis_title="Faixa Etária",
    coloraxis_colorbar=dict(title="Evasão (%)"),
)
fig_14.show()


In [9]:
# ── Gráfico 15: Evasão — Sexo × Turno ────────────────────────────────────────

pivot_15 = heatmap_cruzado(df, "Sexo", "Turno", "Evadidos")
fig_15 = px.imshow(
    pivot_15, text_auto=".1f", color_continuous_scale="Reds",
    labels={"color": "Evasão (%)"},
    title=f"15 — Taxa de Evasão: Sexo × Turno ({ultimo_ano})",
    aspect="auto",
)
fig_15.update_layout(
    xaxis_title="Turno", yaxis_title="Sexo",
    coloraxis_colorbar=dict(title="Evasão (%)"),
)
fig_15.show()


In [10]:
# ── Gráfico 16: Evasão — Cor/Raça × Renda Familiar ───────────────────────────

pivot_16 = heatmap_cruzado(
    df, "Cor / Raça", "Renda Familiar", "Evadidos", ordem_x=ORDEM_RENDA
)
fig_16 = px.imshow(
    pivot_16, text_auto=".1f", color_continuous_scale="Reds",
    labels={"color": "Evasão (%)"},
    title=f"16 — Taxa de Evasão: Cor/Raça × Renda Familiar ({ultimo_ano})",
    aspect="auto",
)
fig_16.update_layout(
    xaxis_title="Renda Familiar (SM per capita)", yaxis_title="Cor/Raça",
    coloraxis_colorbar=dict(title="Evasão (%)"),
)
fig_16.show()


### 4.7. Seção 3 — Heatmaps de retenção (gráficos 17–19)


In [11]:
# ── Gráfico 17: Retenção — Faixa Etária × Renda Familiar ─────────────────────

pivot_17 = heatmap_cruzado(
    df, "Faixa Etária", "Renda Familiar", "Retido",
    ordem_y=ORDEM_ETARIA, ordem_x=ORDEM_RENDA,
)
fig_17 = px.imshow(
    pivot_17, text_auto=".1f", color_continuous_scale="Oranges",
    labels={"color": "Retenção (%)"},
    title=f"17 — Taxa de Retenção: Faixa Etária × Renda Familiar ({ultimo_ano})",
    aspect="auto",
)
fig_17.update_layout(
    xaxis_title="Renda Familiar (SM per capita)",
    yaxis_title="Faixa Etária",
    coloraxis_colorbar=dict(title="Retenção (%)"),
)
fig_17.show()


In [12]:
# ── Gráfico 18: Retenção — Sexo × Turno ──────────────────────────────────────

pivot_18 = heatmap_cruzado(df, "Sexo", "Turno", "Retido")
fig_18 = px.imshow(
    pivot_18, text_auto=".1f", color_continuous_scale="Oranges",
    labels={"color": "Retenção (%)"},
    title=f"18 — Taxa de Retenção: Sexo × Turno ({ultimo_ano})",
    aspect="auto",
)
fig_18.update_layout(
    xaxis_title="Turno", yaxis_title="Sexo",
    coloraxis_colorbar=dict(title="Retenção (%)"),
)
fig_18.show()


In [13]:
# ── Gráfico 19: Retenção — Cor/Raça × Renda Familiar ─────────────────────────

pivot_19 = heatmap_cruzado(
    df, "Cor / Raça", "Renda Familiar", "Retido", ordem_x=ORDEM_RENDA
)
fig_19 = px.imshow(
    pivot_19, text_auto=".1f", color_continuous_scale="Oranges",
    labels={"color": "Retenção (%)"},
    title=f"19 — Taxa de Retenção: Cor/Raça × Renda Familiar ({ultimo_ano})",
    aspect="auto",
)
fig_19.update_layout(
    xaxis_title="Renda Familiar (SM per capita)", yaxis_title="Cor/Raça",
    coloraxis_colorbar=dict(title="Retenção (%)"),
)
fig_19.show()


### 4.8. Seção 4 — Visão consolidada (gráfico 20)


In [14]:
# ── Gráfico 20: Situação × Perfil Sociodemográfico ───────────────────────────
#
# Eixo X: todas as categorias de Sexo, Turno, Renda e Faixa Etária lado a lado.
# Eixo Y: Evasão, Retenção e Conclusão.
# Cor:    percentual daquela situação dentro daquela categoria.
#
# Linhas tracejadas verticais separam as quatro variáveis no eixo X.
# Anotações no topo indicam a qual variável pertence cada grupo de colunas.

def taxa_por_categoria(df, coluna, situacoes):
    """Retorna {label_situacao: Series(categoria -> taxa%)} para todas as situações."""
    total = df.groupby(coluna)["Código da Matricula"].count()
    resultado = {}
    for label, mask in situacoes.items():
        contagem = df[mask].groupby(coluna)["Código da Matricula"].count()
        resultado[label] = (contagem / total * 100).fillna(0).round(1)
    return resultado


situacoes = {
    "Evasão":    df["Categoria da Situação"] == "Evadidos",
    "Retenção":  df["Situação de Matrícula"] == "Retido",
    "Conclusão": df["Categoria da Situação"] == "Concluintes",
}

variaveis_x = [
    ("Sexo",           None),
    ("Turno",          None),
    ("Renda Familiar", ORDEM_RENDA),
    ("Faixa Etária",   ORDEM_ETARIA),
]

colunas_x   = []
separadores = []
matriz      = {s: [] for s in situacoes}

for nome_var, ordem in variaveis_x:
    taxas = taxa_por_categoria(df, nome_var, situacoes)
    cats  = [c for c in (ordem or sorted(df[nome_var].dropna().unique())) if c in df[nome_var].unique()]
    separadores.append(len(colunas_x))
    for cat in cats:
        colunas_x.append(f"{nome_var}: {cat}")
        for sit in situacoes:
            matriz[sit].append(taxas[sit].get(cat, 0))

df_heatmap = pd.DataFrame(matriz, index=colunas_x).T

fig_20 = px.imshow(
    df_heatmap, text_auto=".1f",
    color_continuous_scale="Blues",
    labels={"color": "(%)"},
    title=f"20 — Situação × Perfil Sociodemográfico ({ultimo_ano})",
    aspect="auto",
    height=300,
)
fig_20.update_layout(
    xaxis=dict(tickangle=-45, title=""),
    yaxis=dict(title=""),
    coloraxis_colorbar=dict(title="(%)"),
)

# Linhas divisórias entre variáveis
for pos in separadores[1:]:
    fig_20.add_vline(x=pos - 0.5, line_dash="dash", line_color="gray", line_width=1.5)

# Rótulos de grupo no topo
for i, (nome_var, _) in enumerate(variaveis_x):
    inicio = separadores[i]
    fim    = separadores[i + 1] if i + 1 < len(separadores) else len(colunas_x)
    fig_20.add_annotation(
        x=(inicio + fim - 1) / 2, y=1.12,
        xref="x", yref="paper",
        text=f"<b>{nome_var}</b>",
        showarrow=False, font=dict(size=11),
    )

fig_20.show()
